# Week 48, Day 5: Autocomplete project

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w048d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import heapq

freq = {"python": 90, "pytest": 40, "pyramid": 25, "pie": 60, "pilot": 30, "pyre": 5}

def suggest(prefix, k):
    matches = [(f, w) for w, f in freq.items() if w.startswith(prefix)]
    return [w for f, w in heapq.nlargest(k, matches)]

print(suggest("py", 2), suggest("pi", 3), suggest("q", 2))

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('words.txt', 'w', encoding='utf-8') as f:
    f.write('# word frequency, one per line\npython 90\npytest 40\npyramid 25\npie 60\npilot 30\npyre 5\nPythonic 12\npile 30\npine 18\npizza 75\npizza 1\nplan 55\nplane 20\nplanet 33\nplanner 9\nplay 80\nplayer 44\nplayful 7\nplease 70\npleasant 15\npledge 6\nbad line here\nplot -4\nplum 0\n')

### Exercise: Step 1: loading

Write `load_words(path)`, returning `(freq, bad)`: a dict from word to frequency, and the number of bad lines, following the rules above. Read the file a line at a time.

In [ ]:
def load_words(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_load():\n    \"the example file\"\n    reset_files()\n    freq, bad = load_words(\"words.txt\")\n    assert bad == 3, bad\n    assert freq[\"python\"] == 90 and freq[\"pythonic\"] == 12 and freq[\"pizza\"] == 75 and \"plot\" not in freq and \"plum\" not in freq\n    assert len(freq) == 20\ndef test_edges():\n    \"edge cases\"\n    write_file(\"w.txt\", \"\\n# comment\\n  Apple 3  \\napple 5\\nbanana x\\ncherry 2 extra\\nd4te 9\\n\")\n    assert load_words(\"w.txt\") == ({\"apple\": 5}, 3)\n    write_file(\"e.txt\", \"\")\n    assert load_words(\"e.txt\") == ({}, 0)")

In [ ]:
import heapq

class Rank:
    """Orders (freq, word) so that the WORSE suggestion is smaller."""
    def __init__(self, freq, word):
        self.freq, self.word = freq, word
    def __lt__(self, other):
        if self.freq != other.freq:
            return self.freq < other.freq
        return self.word > other.word

def best_k(pairs, k):
    heap = []
    for word, f in pairs:
        r = Rank(f, word)
        if len(heap) < k:
            heapq.heappush(heap, r)
        elif heap[0] < r:
            heapq.heapreplace(heap, r)
    return [r.word for r in sorted(heap, reverse=True)]

print(best_k([("pile", 30), ("pilot", 30), ("pie", 60), ("pine", 18)], 3))

### Exercise: Step 2: suggestions from a trie

Write `Autocomplete(freq)` (a dict like Step 1's) that builds a trie of `Node`s, and `suggest(prefix, k=5)`: the k best completions, by walking to the prefix's node and visiting its subtree with an explicit stack, keeping a size-k heap. Normalise the prefix (strip, lower-case). An unknown prefix gives `[]`.

In [ ]:
import heapq

class Node:
    def __init__(self):
        self.children = {}
        self.freq = 0

class Rank:
    def __init__(self, freq, word):
        self.freq, self.word = freq, word
    def __lt__(self, other):
        if self.freq != other.freq:
            return self.freq < other.freq
        return self.word > other.word

class Autocomplete:
    def __init__(self, freq):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(freq, prefix, k):\n    prefix = prefix.strip().lower()\n    return sorted((w for w in freq if w.startswith(prefix)), key=lambda w: (-freq[w], w))[:k]\nFREQ = {\"python\": 90, \"pytest\": 40, \"pyramid\": 25, \"pie\": 60, \"pilot\": 30, \"pyre\": 5, \"pythonic\": 12, \"pile\": 30, \"pine\": 18, \"pizza\": 75}\ndef test_suggest():\n    \"suggestions\"\n    a = Autocomplete(FREQ)\n    assert a.suggest(\"py\", 2) == [\"python\", \"pytest\"] and a.suggest(\"pi\", 3) == [\"pizza\", \"pie\", \"pile\"]\n    assert a.suggest(\"  PYTHON \") == [\"python\", \"pythonic\"] and a.suggest(\"q\") == [] and a.suggest(\"p\", 0) == []\ndef test_random():\n    \"against sorting\"\n    rng = random.Random(1)\n    freq = {\"\".join(rng.choice(\"abc\") for _ in range(rng.randint(1, 6))): rng.randint(1, 20) for _ in range(300)}\n    a = Autocomplete(freq)\n    for _ in range(200):\n        p = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 4)))\n        k = rng.randint(1, 8)\n        assert a.suggest(p, k) == slow(freq, p, k), (p, k)")

In [ ]:
class Node:
    def __init__(self):
        self.children, self.freq, self.top = {}, 0, []

K = 3
root = Node()
for word, f in [("pie", 60), ("pizza", 75), ("pile", 30), ("pilot", 30), ("pine", 18)]:
    node = root
    path = [root]
    for ch in word:
        node = node.children.setdefault(ch, Node())
        path.append(node)
    node.freq = f
    for n in path:
        n.top = sorted(n.top + [(f, word)], key=lambda p: (-p[0], p[1]))[:K]

node = root.children["p"].children["i"]
print([w for f, w in node.top])
print([w for f, w in node.children["l"].top])

### Exercise: Step 3: cached suggestions

Write `CachedAutocomplete(freq, K=10)` that keeps a `top` list of at most K `(freq, word)` pairs in each node, best first, built during insertion as described, and `suggest(prefix, k=5)` that only walks to the node and reads its list (k may be at most K; raise `ValueError` for more). The checks count nodes visited per query.

In [ ]:
class Node:
    def __init__(self):
        self.children = {}
        self.freq = 0
        self.top = []

class CachedAutocomplete:
    def __init__(self, freq, K=10):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nclass Counting(dict):\n    gets = 0\n    def get(self, key, default=None):\n        Counting.gets += 1\n        return dict.get(self, key, default)\ndef slow(freq, prefix, k):\n    prefix = prefix.strip().lower()\n    return sorted((w for w in freq if w.startswith(prefix)), key=lambda w: (-freq[w], w))[:k]\ndef test_cached():\n    \"the same answers\"\n    rng = random.Random(2)\n    freq = {\"\".join(rng.choice(\"abc\") for _ in range(rng.randint(1, 6))): rng.randint(1, 20) for _ in range(300)}\n    a = CachedAutocomplete(freq, K=6)\n    for _ in range(300):\n        p = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 4)))\n        k = rng.randint(1, 6)\n        assert a.suggest(p, k) == slow(freq, p, k), (p, k)\n    try:\n        a.suggest(\"a\", 7)\n        assert False\n    except ValueError:\n        pass\n    assert a.suggest(\" A \", 2) == slow(freq, \"a\", 2) and a.suggest(\"zz\", 3) == []\ndef test_lists():\n    \"lists stay short\"\n    a = CachedAutocomplete({\"pie\": 60, \"pizza\": 75, \"pile\": 30, \"pilot\": 30, \"pine\": 18}, K=3)\n    node = a.root.children[\"p\"].children[\"i\"]\n    assert node.top == [(75, \"pizza\"), (60, \"pie\"), (30, \"pile\")]\n    stack = [a.root]\n    while stack:\n        n = stack.pop()\n        assert len(n.top) <= 3\n        stack.extend(n.children.values())")

### Exercise: Step 4: learning

Extend Step 3's engine with `choose(word)` (normalised like a query; raise `ValueError` if it isn't made only of letters), and `frequency(word)`. Keep every cached list correct. The checks make thousands of random choices and compare with a slow version.

In [ ]:
class Node:
    def __init__(self):
        self.children = {}
        self.freq = 0
        self.top = []

class LearningAutocomplete:
    def __init__(self, freq, K=10):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(freq, prefix, k):\n    return sorted((w for w in freq if w.startswith(prefix)), key=lambda w: (-freq[w], w))[:k]\ndef test_learning():\n    \"learning\"\n    a = LearningAutocomplete({\"pie\": 60, \"pizza\": 75, \"pile\": 30}, K=3)\n    assert a.suggest(\"pi\", 2) == [\"pizza\", \"pie\"]\n    for _ in range(31):\n        a.choose(\"pile\")\n    assert a.suggest(\"pi\", 2) == [\"pizza\", \"pile\"] and a.frequency(\"pile\") == 61\n    a.choose(\" Pint \")\n    assert a.frequency(\"pint\") == 1 and a.suggest(\"pin\", 1) == [\"pint\"] and a.frequency(\"zzz\") == 0\n    try:\n        a.choose(\"pi3\")\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(3)\n    freq = {\"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 5))): rng.randint(1, 5) for _ in range(40)}\n    a, ref = LearningAutocomplete(dict(freq), K=4), dict(freq)\n    for _ in range(3000):\n        w = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 5)))\n        a.choose(w)\n        ref[w] = ref.get(w, 0) + 1\n        p = w[:rng.randint(0, len(w))]\n        assert a.suggest(p, 4) == slow(ref, p, 4)")

In [ ]:
def one_edit(s, letters="abcdefghijklmnopqrstuvwxyz"):
    splits = [(s[:i], s[i:]) for i in range(len(s) + 1)]
    deletes = [a + b[1:] for a, b in splits if b]
    swaps = [a + b[1] + b[0] + b[2:] for a, b in splits if len(b) > 1]
    replaces = [a + c + b[1:] for a, b in splits if b for c in letters]
    inserts = [a + c + b for a, b in splits for c in letters]
    return set(deletes + swaps + replaces + inserts) - {s}

edits = one_edit("pyhton")
print(len(edits), "pyhotn" in edits, "python" in edits, "pyton" in edits)

### Exercise: Step 5: one mistake allowed

Write `one_edit(s)` (as above, with lower-case letters) and `fuzzy_suggest(engine, prefix, k)`, using an engine with a `suggest(prefix, k)` method (a simple one is provided). Return up to k words: first the exact prefix's suggestions, then, if there's room, the best suggestions from **all** one-edit prefixes together (ranked by frequency, ties alphabetical, skipping words already listed). Use `engine.freq(word)` for ranking.

In [ ]:
class SimpleEngine:
    def __init__(self, freq):
        self._f = freq

    def freq(self, word):
        return self._f.get(word, 0)

    def suggest(self, prefix, k):
        return sorted((w for w in self._f if w.startswith(prefix)), key=lambda w: (-self._f[w], w))[:k]

def one_edit(s):
    ___

def fuzzy_suggest(engine, prefix, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("FREQ = {\"python\": 90, \"pytest\": 40, \"pyramid\": 25, \"pie\": 60, \"pilot\": 30, \"pyre\": 5, \"pythonic\": 12, \"pizza\": 75, \"play\": 80}\ndef test_edits():\n    \"edits\"\n    e = one_edit(\"ab\")\n    assert \"b\" in e and \"a\" in e and \"ba\" in e and \"cb\" in e and \"abz\" in e and \"ab\" not in e and len(e) == len(set(e))\n    assert len(one_edit(\"pyhton\")) == len({x for x in one_edit(\"pyhton\")})\ndef test_fuzzy():\n    \"suggestions\"\n    eng = SimpleEngine(FREQ)\n    assert fuzzy_suggest(eng, \"pyhton\", 3) == [\"python\", \"pythonic\"]\n    assert fuzzy_suggest(eng, \"py\", 3) == [\"python\", \"pytest\", \"pyramid\"]\n    assert fuzzy_suggest(eng, \"pyth\", 3) == [\"python\", \"pythonic\", \"pytest\"]\n    assert fuzzy_suggest(eng, \"xyz\", 3) == [] and fuzzy_suggest(eng, \"pla\", 1) == [\"play\"]")

### Bug hunt: Best first, or worst first?

This keeps the k best suggestions in a heap but replaces the top when a newcomer is **worse**, so the weakest words keep getting in: this prints `['pizza', 'pine']`. Replace only when the newcomer beats the top, so it prints `['pizza', 'pie']`.

In [ ]:
import heapq

freq = {"pie": 60, "pizza": 75, "pile": 30, "pine": 18}
heap = []
for w, f in freq.items():
    if len(heap) < 2:
        heapq.heappush(heap, (f, w))
    elif (f, w) < heap[0]:
        heapq.heapreplace(heap, (f, w))
print([w for f, w in sorted(heap, reverse=True)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"beat the weakest\"\n    expect_output(\"['pizza', 'pie']\")")

### Bug hunt: A stale cache

When a word's frequency rises, this updates only the word's **own** node's list, not the lists of the nodes above it, so short prefixes keep suggesting the old order: this prints `['pizza']`. Update every node on the path so it prints `['pile']`.

In [ ]:
class Node:
    def __init__(self):
        self.children, self.freq, self.top = {}, 0, []

def put(root, word, f, K=1):
    node, path = root, [root]
    for ch in word:
        node = node.children.setdefault(ch, Node())
        path.append(node)
    node.freq = f
    for n in path[-1:]:
        entries = [p for p in n.top if p[1] != word] + [(f, word)]
        n.top = sorted(entries, key=lambda p: (-p[0], p[1]))[:K]

root = Node()
for w, f in [("pizza", 75), ("pile", 30)]:
    put(root, w, f)
put(root, "pile", 99)
print([w for f, w in root.children["p"].children["i"].top])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the whole path\"\n    expect_output(\"['pile']\")")

### Bug hunt: A word listed twice

When a word's frequency changes, this adds the new entry to the cached list without removing the old one, so the word can appear twice: this prints `['pile', 'pile']`. Remove the word's old entry first so it prints `['pile', 'pizza']`.

In [ ]:
class Node:
    def __init__(self):
        self.children, self.freq, self.top = {}, 0, []

def put(root, word, f, K=2):
    node, path = root, [root]
    for ch in word:
        node = node.children.setdefault(ch, Node())
        path.append(node)
    node.freq = f
    for n in path:
        entries = n.top + [(f, word)]
        n.top = sorted(entries, key=lambda p: (-p[0], p[1]))[:K]

root = Node()
for w, f in [("pizza", 75), ("pile", 80), ("pile", 99)]:
    put(root, w, f)
print([w for f, w in root.top])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no repeats\"\n    expect_output(\"['pile', 'pizza']\")")

### Puzzle: Updating the cached lists

Arrange the lines to set a word's frequency and update every cached list on its path. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# The engine holds: pie 60, pizza 75, pile 30, pilot 30, pine 18, with K = 3.

p = ___   # suggest("pi", 3)
q = ___   # suggest("pil", 3)
r = ___   # the cached top list at the node for "pin", as (freq, word) pairs
s = ___   # suggest("pi", 3) after choose("pine") is called 50 times
t = ___   # True or False: "pyhton" and "python" are one edit apart

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [\"pizza\", \"pie\", \"pile\"] and q == [\"pile\", \"pilot\"] and r == [(18, \"pine\")], \"p, q and r\"\n    assert s == [\"pizza\", \"pine\", \"pie\"] and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the finished engine

Put it together as one class, `Engine`, with a small command interface:

- `Engine.from_file(path, K=10)`: load with Step 1's rules (keep `bad`, the number of bad lines, as an attribute) and build the cached trie.
- `suggest(prefix, k=5)`: cached suggestions (Step 3); if there are fewer than k, fill up with one-edit corrections (Step 5) ranked by frequency then alphabetically, never repeating a word.
- `choose(word)`: as in Step 4.
- `handle(line)`: one command as text, returning the reply as a string. `"? pre"` replies with the suggestions joined by spaces (or `"(none)"`); `"! word"` chooses the word and replies `"ok"`; anything else, including a bad word, replies `"error: ..."` with a short message, never raising.
- `save(path)`: write the current frequencies in the Step 1 format, one `word frequency` line per word in alphabetical order, so `from_file` can reload them.

The checks load the example file, run a session of commands, save, reload, and compare.

In [ ]:
class Node:
    def __init__(self):
        self.children = {}
        self.freq = 0
        self.top = []

class Engine:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_session():\n    \"a session\"\n    reset_files()\n    e = Engine.from_file(\"words.txt\", K=5)\n    assert e.bad == 3\n    assert e.handle(\"? py\") == \"python pytest pyramid pythonic pyre\"\n    assert e.handle(\"? pyhton\") == \"python pythonic\"\n    assert e.handle(\"? zzzz\") == \"(none)\"\n    assert e.handle(\"! pyre\") == \"ok\"\n    for _ in range(40):\n        e.handle(\"! pyre\")\n    assert e.handle(\"? py\").split()[:2] == [\"python\", \"pyre\"]\n    assert e.handle(\"! py3\").startswith(\"error\") and e.handle(\"nonsense\").startswith(\"error\") and e.handle(\"\").startswith(\"error\")\n    e.save(\"saved.txt\")\n    lines = read_file(\"saved.txt\").splitlines()\n    assert lines[0] == \"pie 60\" and \"pyre 46\" in lines and lines == sorted(lines, key=lambda l: l.split()[0])\n    again = Engine.from_file(\"saved.txt\", K=5)\n    assert again.bad == 0 and again.handle(\"? py\") == e.handle(\"? py\") and again.handle(\"? pl\") == e.handle(\"? pl\")")

## The plan

That's autocomplete in four lines, and for six words it's perfect. For a million words, every key press would scan all million. The rest of today makes the same answers come back in time proportional to what you typed, not to the size of the dictionary.

The engine's behaviour, which every step keeps:

- Suggestions for a prefix are the words that **start with** it, most frequent first, **ties alphabetical**, at most k of them.
- Words are stored in lower case; queries are lower-cased and stripped of surrounding spaces before use.
- The word itself counts as a completion of its own prefix: typing "python" suggests "python" first.

## Step 1: loading the words

The word list comes from a file you didn't write, so the loader checks every line (Week 37, Week 46 Day 5). Each useful line is `word frequency`. Lines that are blank or start with `#` are skipped quietly. A line is **bad** (counted, then skipped) if it doesn't have exactly two parts, the word isn't made only of letters, or the frequency isn't a whole number greater than 0. A word listed twice gets the **larger** frequency. Words are lower-cased.

## Step 2: top suggestions

Put the words in a trie whose nodes hold the frequency of the word ending there (0 if none). For a prefix, walk to its node, then visit the subtree, keeping the best k with a heap of size k (Day 4 of last week): that's O(L + size of subtree × log k). A short prefix like "p" still visits a large subtree, but a longer one is already much faster than scanning every word.

"Best" means higher frequency, then alphabetically earlier. To keep the k best in a min-heap whose top is the **worst** kept, the heap's order must put worse entries first: lower frequency first, and among equal frequencies, the alphabetically **later** word first. Python can't negate a string, so a small wrapper class with its own `__lt__` keeps it clear:

## Step 3: answers on every key press

Visiting the subtree on every key press is still too slow for a big dictionary and a one-letter prefix. Trade memory for time: store in **every node** a short list, `top`, of the best K completions passing through it (K being the most suggestions the engine will ever show, say 10). Then a query walks to the node and reads its list: **O(L + k)**, however big the subtree.

The lists are built as words are inserted: for each node on the new word's path, add the word to that node's list, re-sort by rank, and cut it back to K. That's O(L × K log K) per insertion, done once, rather than on every key press. Memory grows by up to K entries per node, which is why K stays small.

## Step 4: learning from choices

When someone picks a suggestion, it should rank higher next time. `choose(word)` adds 1 to the word's frequency (adding the word with frequency 1 if it's new). Because frequencies only ever **go up**, the cached lists stay correct if every node on the word's path updates its own list: a word can only enter a node's top K by rising, and any node it rises in is on its path. (If frequencies could go **down**, a word dropping out of a list would need a replacement from elsewhere in the subtree, and the cache would need rebuilding from the subtree.)

## Step 5: forgiving a typo

People mistype. If "pyhton" has no completions, the engine should still offer "python". A simple, effective approach comes from Peter Norvig's essay "How to Write a Spelling Corrector": generate every string **one edit** away from what was typed (delete a character, swap two neighbours, replace a character, or insert one), and look each one up. For a prefix of length L over 26 letters that's about 54L + 25 candidates, and each costs one O(L + k) cached lookup.

Ranking: completions of the exact prefix come first (the user may not have made a mistake at all); then the completions of the edited prefixes, best first, without repeats.

## Checkpoint and quiz

## Daily challenge

## Before you go

Type slowly into a search box on your phone and watch the suggestions change with each letter. Which of today's ideas do you think it uses, and what does it do that yours doesn't (context, your own history, other languages)? Tomorrow is the Forge.